In [1]:
import torch
import os

import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, quantize_static, CalibrationDataReader, QuantType

import numpy as np

from PieceDetection.PieceDetection_CNN.PieceDetection_CNN import PieceDetectorCNNModel

# Create Onnx

In [10]:
model = PieceDetectorCNNModel()
model.load_state_dict(torch.load(os.path.join(os.environ["WEIGHTS"], "piece_cnn.pt")))

/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


<All keys matched successfully>

In [11]:
dummy_input = torch.randn(1,8,8,3,128,64)
torch.onnx.export(model, dummy_input, "piece_cnn.onnx", opset_version=13)

/mnt/D/University/Thesis/src/PieceDetection/PieceDetection_CNN/PieceDetection_CNN.py:176: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if C != 3:
/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/torch/onnx/symbolic_opset11.py:903: UserWarning: This model contains a squeeze operation on dimension 1. The size of this dimension in the given input is 512. The model will be exported without the squeeze node. If the model is intended to be used with dynamic input shapes, please export with dynamic_axes argument.
  warnings.warn(


# Optimize Onnx (Dynamic Quantization)

In [12]:
model_onnx = onnx.load("piece_cnn.onnx")
onnx.checker.check_model(model_onnx)

In [13]:
model_fp32 = "piece_cnn.onnx"
model_int8 = "piece_cnn_quantized_dynamic.onnx"

quantize_dynamic(
    model_input=model_fp32,
    model_output=model_int8,
    weight_type=QuantType.QUInt8,  # or QuantType.QUInt8
    # op_types_to_quantize=["MatMul", "Gemm"]
)

# Load and Test Onnx

In [9]:
print(ort.get_available_providers())
session = ort.InferenceSession(
    "piece_cnn.onnx",
    providers=["CPUExecutionProvider"]
    # providers=["CUDAExecutionProvider"]

    # "piece_cnn_quantized_dynamic.onnx",
    # "piece_cnn_quantized_static.onnx",
    # providers=["OpenVINOExecutionProvider"]
)
print(session.get_providers())

input_name = session.get_inputs()[0].name
output_names = [session.get_outputs()[i].name for i in range(len(session.get_outputs()))]

input_name, output_names

['OpenVINOExecutionProvider', 'CPUExecutionProvider']
['CPUExecutionProvider']


('onnx::Reshape_0', ['271', '275', '289', 'onnx::Gemm_253'])

In [10]:
x = np.random.randn(1,8,8,3,64,64).astype(np.float32)

In [11]:
%%timeit
outputs = session.run(output_names, {input_name: x})

78.6 ms ± 1.69 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
outputs = session.run(output_names, {input_name: x})
outputs[0].shape, outputs[1].shape, outputs[2].shape, outputs[3].shape

((1, 8, 8), (1, 8, 8), (1, 8, 8, 6), (64, 512))

# Optimize Onnx (Static Quantization)

In [14]:
from Dataset.DataSetLoaders import ChessDataset
from PieceDetection.PieceCropper_3D import PieceCropper

In [15]:
ds = ChessDataset.ChessDataset(
    config={
        "img_size": (640,640)
    }
)

def prep(entry):
    img, lbl = entry
    PieceCropper.piece_cropper.set_img(img, lbl["corners"])
    board_split = PieceCropper.piece_cropper.process_img()
    return board_split.unsqueeze(0)


class CalibrationDataReader(CalibrationDataReader):
    def __init__(self, input_name):
        self.input_name = input_name
        self.data_iter = iter([
            {input_name: prep(ds[_]).numpy()}
            for _ in range(10)
        ])

    def get_next(self):
        return next(self.data_iter, None)

In [16]:
# load model to find input name
model_fp32 = "piece_cnn.onnx"
session = ort.InferenceSession(model_fp32, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

dr = CalibrationDataReader(input_name)

quantize_static(
    model_input=model_fp32,
    model_output="piece_cnn_quantized_static.onnx",
    calibration_data_reader=dr,
    weight_type=QuantType.QInt8,
)
print("✅ Static quantization done")

✅ Static quantization done
